In [ ]:
!pip install yfinance pandas tabulate tqdm -q

In [ ]:
import yfinance as yf
import pandas as pd
import time
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url)
    tickers = tables[0]['Symbol'].tolist()
    return [t.replace('.', '-') if '.' in t else t for t in tickers]

BATCH_SIZE = 25
partia = 1                    # ←←← ZMIENIAJ TYLKO TĘ LICZBĘ (1, 2, 3...)

tickers = get_sp500_tickers()
total_batches = (len(tickers) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"✅ Załadowano {len(tickers)} spółek z S&P500")
print(f"📦 Partia {partia} z {total_batches} → zostało {total_batches - partia} kliknięć")

start = (partia - 1) * BATCH_SIZE
current_batch = tickers[start : start + BATCH_SIZE]

data = []
for t in tqdm(current_batch):
    try:
        info = yf.Ticker(t).info
        if not info.get('regularMarketPrice'): continue
        def safe(v, d=1):
            try: return round(float(v)/d, 2) if v else None
            except: return None
        data.append({
            'Ticker': t,
            'PE': info.get('trailingPE'),
            'PE_fwd': info.get('forwardPE'),
            'PEG': info.get('pegRatio'),
            'PB': info.get('priceToBook'),
            'PS': info.get('priceToSalesTrailing12Months'),
            'EV_EBITDA': info.get('enterpriseToEbitda'),
            'ROE': safe(info.get('returnOnEquity'), 0.01),
            'RevGrowth': safe(info.get('revenueGrowth'), 0.01)
        })
    except:
        pass
    time.sleep(0.7)

df = pd.DataFrame(data)
if not df.empty:
    for c in ['PE','PE_fwd','PEG','PB','PS','EV_EBITDA']:
        if c in df.columns:
            m = df[c].median()
            if m and m > 0:
                df[c + '_vsMed'] = (df[c] / m).round(2)

    print("\n" + "="*100)
    print(f"TABELA — PARTIA {partia}/{total_batches}")
    print("="*100)
    print(df.to_markdown(index=False))

    print("\n" + "="*90)
    print("SKOPIUJ PROMPT I WKLEJ DO GROK")
    print("="*90)
    prompt = f"""Przeanalizuj partię {partia} z S&P500.

{df.to_markdown(index=False)}"""
    print(prompt)
